# 04 Train NER Transformer

This notebook trains the first entity-recognition baseline:

```text
BERT token classification + BIO tagging
```

Run `03_prepare_radgraph_ner_re_dataset.ipynb` first. This notebook does not display raw report text. Model files and predictions are written under `outputs/`, which is ignored by Git.

Progress is shown during data loading, tokenisation, training, evaluation, and prediction. Set `RUN_SMOKE_TEST = True` in the configuration cell before a full experiment to verify the environment quickly.


In [ ]:
from __future__ import annotations

import inspect
import json
import math
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset, DatasetDict
from dotenv import load_dotenv
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from tqdm.auto import tqdm
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    set_seed,
)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing pyproject.toml")


def format_duration(seconds: float) -> str:
    seconds = max(0, int(seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

HF_HOME = os.environ.get("HF_HOME")
if HF_HOME:
    os.environ.setdefault("HF_HOME", HF_HOME)

RUN_NAME = os.getenv("RADGRAPH_XL_RUN_NAME", "full_2300")
RUN_ROOT = PROJECT_ROOT / "outputs" / RUN_NAME
INTERIM_DIR = RUN_ROOT / "interim"
RESULTS_DIR = RUN_ROOT / "results"
PREDICTIONS_DIR = RUN_ROOT / "predictions"
MODEL_DIR = RUN_ROOT / "models" / "ner_bert_base_uncased"

for path in [RESULTS_DIR, PREDICTIONS_DIR, MODEL_DIR]:
    path.mkdir(parents=True, exist_ok=True)

NER_JSONL = INTERIM_DIR / "ner_dataset.jsonl"
LABEL_MAPS_JSON = INTERIM_DIR / "label_maps.json"

assert NER_JSONL.exists(), "Run 03_prepare_radgraph_ner_re_dataset.ipynb before this notebook."
assert LABEL_MAPS_JSON.exists(), "Missing label_maps.json. Run notebook 03 first."

# Hugging Face model used as the general-domain baseline. Keep all other settings
# identical when comparing this model with PubMedBERT or BioClinicalBERT.
MODEL_CHECKPOINT = "bert-base-uncased"

# Maximum subword sequence length. BERT cannot exceed 512; longer reports require
# sliding windows rather than a larger value. Candidate window experiment: 256 vs 512.
MAX_LENGTH = 512

# AdamW peak learning rate. Larger values learn faster but can destabilise fine-tuning.
# Recommended controlled search: 1e-5, 2e-5, 3e-5, then optionally 5e-5.
LEARNING_RATE = 2e-5

# Maximum passes over the training set. Compare 3, 5, and 6 with validation F1 and
# early stopping; do not select an epoch count using the test set.
NUM_EPOCHS = 3

# Reports processed per GPU step. Increase only to improve throughput or stabilise
# gradients; it does not automatically improve F1. Try 4 and 8 if VRAM permits.
TRAIN_BATCH_SIZE = 4

# Evaluation-only batch size. Increase until GPU memory is nearly full; this changes
# speed but not model learning.
EVAL_BATCH_SIZE = 8

# AdamW regularisation strength. 0.01 is the standard baseline; test 0.0 and 0.01.
WEIGHT_DECAY = 0.01

# Learning-rate schedule. Linear is the Hugging Face default and preserves the baseline.
LR_SCHEDULER_TYPE = "linear"

# Fraction of training steps used to ramp the learning rate from zero. Keep 0.0 for
# exact baseline reproduction; test 0.1 as a separate scheduling experiment.
WARMUP_RATIO = 0.0

# Number of mini-batches accumulated before an optimiser update. Use 2 or 4 when a
# larger effective batch is needed but GPU memory cannot hold it directly.
GRADIENT_ACCUMULATION_STEPS = 1

# Gradient clipping threshold. Lower values can stabilise spikes; 1.0 is standard.
MAX_GRAD_NORM = 1.0

# Reproducibility seed. Use 42 for tuning, then repeat final settings with 13 and 21.
RANDOM_SEED = 42

# Frequency of durable progress messages. Smaller values show more detail but add I/O.
LOGGING_STEPS = 10

# Run this first. It trains on a tiny subset and usually finishes within a few minutes.
RUN_SMOKE_TEST = False

# Keep these as None for the full experiment. Smoke-test mode overrides them below.
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None
MAX_TEST_SAMPLES = None

if RUN_SMOKE_TEST:
    NUM_EPOCHS = 1
    MAX_TRAIN_SAMPLES = 16
    MAX_EVAL_SAMPLES = 8
    MAX_TEST_SAMPLES = 8

set_seed(RANDOM_SEED)

environment = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "project_root": str(PROJECT_ROOT),
    "python_executable": sys.executable,
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "torch_cuda_version": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu_count": torch.cuda.device_count(),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "model_checkpoint": MODEL_CHECKPOINT,
    "run_smoke_test": RUN_SMOKE_TEST,
}

print("Environment check")
for key, value in environment.items():
    print(f"  {key}: {value}")

if not torch.cuda.is_available():
    print("\nWARNING: CUDA is unavailable. Stop here and select the the project's .venv kernel before training.")


In [ ]:
def load_jsonl(path: Path) -> list[dict]:
    with path.open("r", encoding="utf-8") as handle:
        total_lines = sum(1 for line in handle if line.strip())

    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line in tqdm(handle, total=total_lines, desc="Loading NER reports", unit="report"):
            if line.strip():
                rows.append(json.loads(line))
    return rows


load_started = time.perf_counter()
rows = load_jsonl(NER_JSONL)
label_maps = json.loads(LABEL_MAPS_JSON.read_text(encoding="utf-8"))

label_to_id = {label: int(idx) for label, idx in label_maps["ner_label_to_id"].items()}
id_to_label = {int(idx): label for idx, label in label_maps["ner_id_to_label"].items()}

splits = {
    split: [row for row in rows if row["split"] == split]
    for split in ["train", "validation", "test"]
}

if MAX_TRAIN_SAMPLES is not None:
    splits["train"] = splits["train"][:MAX_TRAIN_SAMPLES]
if MAX_EVAL_SAMPLES is not None:
    splits["validation"] = splits["validation"][:MAX_EVAL_SAMPLES]
if MAX_TEST_SAMPLES is not None:
    splits["test"] = splits["test"][:MAX_TEST_SAMPLES]

dataset = DatasetDict({split: Dataset.from_list(items) for split, items in splits.items()})

print(f"Loaded and split data in {format_duration(time.perf_counter() - load_started)}")
print({split: len(items) for split, items in splits.items()})
print({"num_labels": len(label_to_id), "labels": list(label_to_id)})


In [ ]:
tokenizer_started = time.perf_counter()
print(f"Loading tokenizer: {MODEL_CHECKPOINT}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)
print(f"Tokenizer ready in {format_duration(time.perf_counter() - tokenizer_started)}")


def tokenize_and_align_labels(batch: dict) -> dict:
    tokenized = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    aligned_labels = []
    for batch_index, bio_labels in enumerate(batch["bio_labels"]):
        word_ids = tokenized.word_ids(batch_index=batch_index)
        previous_word_id = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(label_to_id[bio_labels[word_id]])
            else:
                # Ignore continuation subwords to avoid overweighting split tokens.
                label_ids.append(-100)
            previous_word_id = word_id
        aligned_labels.append(label_ids)

    tokenized["labels"] = aligned_labels
    return tokenized


tokenisation_started = time.perf_counter()
tokenized_splits = {}
for split_name, split_dataset in dataset.items():
    split_started = time.perf_counter()
    tokenized_splits[split_name] = split_dataset.map(
        tokenize_and_align_labels,
        batched=True,
        remove_columns=split_dataset.column_names,
        desc=f"Tokenising {split_name}",
    )
    print(
        f"Finished {split_name}: {len(split_dataset)} reports in "
        f"{format_duration(time.perf_counter() - split_started)}"
    )

tokenized_dataset = DatasetDict(tokenized_splits)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

print(f"All splits tokenised in {format_duration(time.perf_counter() - tokenisation_started)}")
print(tokenized_dataset)


In [ ]:
def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    true_predictions = []
    true_labels = []
    for pred_row, label_row in zip(predictions, labels):
        pred_labels = []
        gold_labels = []
        for pred_id, label_id in zip(pred_row, label_row):
            if label_id == -100:
                continue
            pred_labels.append(id_to_label[int(pred_id)])
            gold_labels.append(id_to_label[int(label_id)])
        true_predictions.append(pred_labels)
        true_labels.append(gold_labels)

    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }


In [ ]:
class ProgressPrinterCallback(TrainerCallback):
    # Print durable progress messages even if the notebook progress bar is not rendered.

    def __init__(self) -> None:
        self.started_at = None
        self.last_logged_step = -1

    def on_train_begin(self, args, state, control, **kwargs):
        self.started_at = time.perf_counter()
        print(
            f"Training started: {state.max_steps} optimizer steps, "
            f"{args.num_train_epochs:g} epoch(s), device={args.device}"
        )

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs or state.global_step == self.last_logged_step:
            return
        self.last_logged_step = state.global_step
        elapsed = time.perf_counter() - self.started_at if self.started_at is not None else 0
        total = max(1, state.max_steps)
        percent = 100 * state.global_step / total
        values = []
        for key in ["loss", "eval_loss", "eval_f1", "learning_rate"]:
            if key in logs:
                value = logs[key]
                values.append(f"{key}={value:.6g}" if isinstance(value, (int, float)) else f"{key}={value}")
        suffix = ", ".join(values)
        print(
            f"[progress] step {state.global_step}/{state.max_steps} "
            f"({percent:.1f}%), epoch={state.epoch or 0:.2f}, "
            f"elapsed={format_duration(elapsed)}" + (f", {suffix}" if suffix else "")
        )

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        metrics = metrics or {}
        summary = {
            key: round(value, 4) if isinstance(value, (int, float)) else value
            for key, value in metrics.items()
            if key in {"eval_loss", "eval_precision", "eval_recall", "eval_f1"}
        }
        print(f"Validation completed at step {state.global_step}: {summary}")

    def on_train_end(self, args, state, control, **kwargs):
        elapsed = time.perf_counter() - self.started_at if self.started_at is not None else 0
        print(f"Training finished in {format_duration(elapsed)}")


model_started = time.perf_counter()
print(f"Loading model: {MODEL_CHECKPOINT}")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(label_to_id),
    id2label=id_to_label,
    label2id=label_to_id,
)
print(f"Model ready in {format_duration(time.perf_counter() - model_started)}")

args_kwargs = {
    "output_dir": str(MODEL_DIR),
    "learning_rate": LEARNING_RATE,
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "num_train_epochs": NUM_EPOCHS,
    "weight_decay": WEIGHT_DECAY,
    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "warmup_ratio": WARMUP_RATIO,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "max_grad_norm": MAX_GRAD_NORM,
    "save_strategy": "epoch",
    "save_total_limit": 2,
    "logging_strategy": "steps",
    "logging_steps": LOGGING_STEPS,
    "logging_first_step": True,
    "disable_tqdm": False,
    "log_level": "info",
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "report_to": [],
    "seed": RANDOM_SEED,
    "dataloader_num_workers": 0,
}

signature = inspect.signature(TrainingArguments.__init__)
if "eval_strategy" in signature.parameters:
    args_kwargs["eval_strategy"] = "epoch"
else:
    args_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**args_kwargs)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[ProgressPrinterCallback()],
)

steps_per_epoch = math.ceil(len(tokenized_dataset["train"]) / TRAIN_BATCH_SIZE)
estimated_steps = steps_per_epoch * NUM_EPOCHS
print(
    "Training plan:",
    {
        "train_reports": len(tokenized_dataset["train"]),
        "validation_reports": len(tokenized_dataset["validation"]),
        "epochs": NUM_EPOCHS,
        "batch_size": TRAIN_BATCH_SIZE,
        "estimated_optimizer_steps": estimated_steps,
        "logging_every_steps": LOGGING_STEPS,
        "device": str(training_args.device),
    },
)

if training_args.device.type != "cuda":
    raise RuntimeError(
        "Training would run on CPU. Select the the project's .venv Jupyter kernel and rerun from the first cell."
    )

print("Starting trainer.train() at", datetime.now().isoformat(timespec="seconds"))
train_started = time.perf_counter()
train_result = trainer.train()
print("trainer.train() returned after", format_duration(time.perf_counter() - train_started))
print(train_result.metrics)

if torch.cuda.is_available():
    print(
        "Peak GPU memory allocated (GiB):",
        round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2),
    )


In [ ]:
print("Starting test-set evaluation")
evaluation_started = time.perf_counter()
test_metrics = trainer.evaluate(tokenized_dataset["test"], metric_key_prefix="test")
print("Evaluation finished in", format_duration(time.perf_counter() - evaluation_started))

print("Saving best model and tokenizer")
trainer.save_model(str(MODEL_DIR / "best_model"))
tokenizer.save_pretrained(str(MODEL_DIR / "best_model"))

metrics_path = RESULTS_DIR / "ner_bert_base_uncased_metrics.json"
metrics_path.write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")

print(json.dumps(test_metrics, indent=2))
print("Saved metrics:", metrics_path)


In [ ]:
def bio_to_spans(labels: list[str]) -> list[dict]:
    spans = []
    start = None
    current_label = None

    def close_span(end_index: int):
        if start is not None and current_label is not None:
            spans.append({"start": start, "end": end_index, "label": current_label})

    for idx, label in enumerate(labels):
        if label == "O":
            if current_label is not None:
                close_span(idx - 1)
                start = None
                current_label = None
            continue
        prefix, entity_label = label.split("-", 1)
        if prefix == "B" or current_label != entity_label:
            if current_label is not None:
                close_span(idx - 1)
            start = idx
            current_label = entity_label
        # I-label continuing the same entity does not need action.

    if current_label is not None:
        close_span(len(labels) - 1)

    return spans


def predict_word_labels(row: dict) -> list[str]:
    encoded = tokenizer(
        row["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    word_ids = encoded.word_ids(batch_index=0)
    encoded = {key: value.to(model.device) for key, value in encoded.items()}
    model.eval()
    with torch.no_grad():
        logits = model(**encoded).logits[0].detach().cpu().numpy()
    pred_ids = logits.argmax(axis=-1)
    word_predictions = {}
    for token_index, word_id in enumerate(word_ids):
        if word_id is None or word_id in word_predictions:
            continue
        word_predictions[word_id] = id_to_label[int(pred_ids[token_index])]
    return [word_predictions.get(idx, "O") for idx in range(len(row["tokens"]))]


prediction_path = PREDICTIONS_DIR / "ner_bert_base_uncased_test_predictions.jsonl"
prediction_started = time.perf_counter()
with prediction_path.open("w", encoding="utf-8") as handle:
    for row in tqdm(splits["test"], desc="Generating span predictions", unit="report"):
        predicted_labels = predict_word_labels(row)
        output = {
            "source": row.get("source"),
            "dataset": row.get("dataset"),
            "doc_id": row["doc_id"],
            "split": row["split"],
            "predicted_entities": bio_to_spans(predicted_labels),
            "gold_entities": bio_to_spans(row["bio_labels"]),
        }
        handle.write(json.dumps(output, ensure_ascii=False) + "\n")

print("Prediction export finished in", format_duration(time.perf_counter() - prediction_started))
print("Saved predictions:", prediction_path)


In [ ]:
print("Generating the detailed test classification report")
report_started = time.perf_counter()
predictions = trainer.predict(tokenized_dataset["test"])
pred_ids = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

true_predictions = []
true_labels = []
for pred_row, label_row in tqdm(
    zip(pred_ids, labels),
    total=len(labels),
    desc="Formatting test labels",
    unit="report",
):
    pred_labels = []
    gold_labels = []
    for pred_id, label_id in zip(pred_row, label_row):
        if label_id == -100:
            continue
        pred_labels.append(id_to_label[int(pred_id)])
        gold_labels.append(id_to_label[int(label_id)])
    true_predictions.append(pred_labels)
    true_labels.append(gold_labels)

report = classification_report(true_labels, true_predictions, digits=4)
report_path = RESULTS_DIR / "ner_bert_base_uncased_classification_report.txt"
report_path.write_text(report, encoding="utf-8")

print(report)
print("Report generation finished in", format_duration(time.perf_counter() - report_started))
print("Saved report:", report_path)
